# Map drawn in the photo's own view, above 1000 m (iteration 43)

**Idea (user, 2026-11-18).** Today the *photo* is warped onto a north-up map canvas of 224 px (a canvas pixel is about 2x the photo pixel at 1400 m, and 9 to 26 % of the photo's pixels fall outside the canvas). Turn it round: leave the photo as it is (320 x 256, native resolution, only cleaned) and draw the *map* into the photo's camera view with the photo's pose (the same projection the synthetic audit used), at a map level whose pixel is about the photo's ground pixel. The matcher then sees the photo untouched and a map that already has its perspective, tilt and heading.

**What is measured (true-place diagnostic with a bar from wrong places, as in iteration 36).** For every working photo above 1000 m: the best inlier count over 3 heading queries at the TRUE place, and at 4 WRONG places 3 km away (same pose, same projection). Inputs: `canvas` (today's input), `photo view` (320 x 256 photo against the map drawn in its view) and `photo view x2` (both enlarged 2x, so the matcher's 8-pixel grid is finer). Matchers: LoFTR and XoFTR (MINIMA weights). Each input and matcher gets its own bar from its own wrong-place counts (cross-validated over odd/even photos); recall = share of photos whose true-place count beats the bar; AUC is threshold-free.

**Kill rule (written before the run).** A photo-view input is useful only if, for the same matcher, its cross-validated recall at the zero-false-positive bar is at least 10 points above the canvas input in at least two of the three bands (1000-1300, 1300-1600, 1600-2100 m). If useful, the next step is wiring it into the pipeline's fine stage; if not, the idea is dead. Caveat written first: the pose has the metadata's noise (about 1 degree attitude, 2 % height), so the map drawn in the photo's view is off by a small unknown warp; the similarity fit and the matcher have to absorb it. A tilt-corrected map cannot be better than the pose it is drawn with.

Env: `EF_HEADLESS, EF_DATASET, EF_STORE, EF_OUT, EF_TAG, EF_LIMIT, EF_MATCHERS, EF_EVERY, EF_MIN_HEIGHT`.

In [1]:
import os, sys, math, colorsys
import numpy as np, pandas as pd, torch, cv2
HEADLESS = os.environ.get("EF_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [int(v) for v in os.environ.get("EF_LEVELS", "3,4,5,6,7,8,9,10,11").split(",")]
from geoloc import photo, queries, scoring, fine_matching, flat_field
from geoloc.cli import read_rows, move_hint
from geoloc.coarse_search import DEVICE, score_map_for_query, pick_peaks, window_transforms, zncc_everywhere, fast_length, cut_map_features
from geoloc.edge_features import normalise_cells
from geoloc.levels import level_for_height, level_gsd, map_piece
from geoloc.map_store import MapStore
CELL = settings.CELL; TILE = settings.TILE
DATASET = os.environ["EF_DATASET"]; STORE = os.environ["EF_STORE"]; OUT = os.environ.get("EF_OUT", "results/edge_features"); os.makedirs(OUT, exist_ok=True)
store = MapStore(STORE); rows = read_rows(DATASET)       # the dataset folder holds the development photos only
heights = np.array([float(r["alt"]) for r in rows]); print(len(rows), "photos, heights", heights.min(), heights.max())

332 photos, heights 100.599998 2016.599976


In [2]:
import time, sys
from PIL import Image
LIMIT = int(os.environ.get("EF_LIMIT", 0)); MATCHERS = os.environ.get("EF_MATCHERS", "loftr,xoftr,roma").split(",")
MODELS = os.path.join(os.path.abspath(HEADLESS), "models"); CKPT = os.environ.get("EF_CKPT", "/mnt/e/IdeaForge/work/NUDT/checkpoint")
ORIGINAL_NORMALISE = photo.normalise_frame
engines = {}
loftr = fine_matching.load_loftr(os.path.join(MODELS, settings.LOFTR_FILE))
def loftr_pairs(a, b):
    pa, pb, conf = fine_matching.loftr_matches(loftr, a, b); keep = conf >= settings.LOFTR_CONFIDENCE; return pa[keep], pb[keep]
engines["loftr"] = loftr_pairs
if "xoftr" in MATCHERS:
    THIRD = os.environ.get("EF_XOFTR", "/mnt/e/IdeaForge/work/repo/third_party/XoFTR"); sys.path.insert(0, THIRD)
    from xoftr import XoFTR
    def xoftr_config(coarse, fine):
        return dict(resolution=(8, 2), fine_window_size=5, medium_window_size=3, resnet=dict(initial_dim=128, block_dims=[128, 196, 256]),
                    coarse=dict(inference=True, d_model=256, d_ffn=256, nhead=8, layer_names=["self", "cross"] * 4, attention="linear"),
                    match_coarse=dict(inference=True, d_model=256, thr=coarse, border_rm=2, match_type="dual_softmax", dsmax_temperature=0.1, train_coarse_percent=0.2, train_pad_num_gt_min=200),
                    fine=dict(denser=False, inference=True, dsmax_temperature=0.1, thr=fine, mlp_hidden_dim_coef=2, nhead_fine_level=8, nhead_medium_level=7))
    saved = torch.load(os.path.join(CKPT, "MINIMA", "minima_xoftr.ckpt"), map_location="cpu", weights_only=False)
    weights = {k[len("matcher."):]: v for k, v in saved["state_dict"].items() if k.startswith("matcher.")}
    xmodel = XoFTR(config=xoftr_config(0.3, 0.1)); xmodel.load_state_dict(weights, strict=True); xmodel = xmodel.to(DEVICE).eval()
    @torch.no_grad()
    def xoftr_pairs(a, b):
        batch = {"image0": fine_matching.as_tensor(a), "image1": fine_matching.as_tensor(b)}; xmodel(batch)
        return batch["mkpts0_f"].cpu().numpy(), batch["mkpts1_f"].cpu().numpy()
    engines["xoftr"] = xoftr_pairs
if "roma" in MATCHERS:
    from romatch import roma_outdoor
    rweights = torch.load(os.path.join(CKPT, "MINIMA", "minima_roma.pth"), map_location=DEVICE); dino = torch.load(os.path.join(CKPT, "RoMa", "dinov2_vitl14_pretrain.pth"), map_location=DEVICE)
    roma = roma_outdoor(DEVICE, weights=rweights, dinov2_weights=dino, coarse_res=560, upsample_res=(672, 672), amp_dtype=torch.float16, use_custom_corr=False).eval()
    def to_pil(g): u = np.clip(np.asarray(g, np.float32) * 255, 0, 255).astype(np.uint8); return Image.fromarray(np.stack([u, u, u], -1))
    @torch.no_grad()
    def roma_pairs(a, b):
        torch.manual_seed(0); warp, cert = roma.match(to_pil(a), to_pil(b), device=DEVICE); m, _ = roma.sample(warp, cert, num=2000)
        ha, wa = a.shape; hb, wb = b.shape; pa, pb = roma.to_pixel_coordinates(m, ha, wa, hb, wb); return pa.cpu().numpy().astype(np.float32), pb.cpu().numpy().astype(np.float32)
    engines["roma"] = roma_pairs
print("matchers:", list(engines))

/home/workstation/training/ideaforge/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


matchers: ['loftr', 'xoftr']


In [3]:
def with_settings(flat=None, degree=3, local=None):
    settings.FLAT_FIELD_FROM_HEIGHT_M = flat; settings.FLAT_FIELD_DEGREE = degree; settings.LOCAL_NORM_SIGMA = local
def clean_tile(tile, sigma):
    has = tile > 0; fill = float(tile[has].mean()) if has.any() else 0.5
    normal = flat_field.local_normalise(np.where(has, tile, fill).astype(np.float32), sigma); out = ORIGINAL_NORMALISE(normal)
    return np.where(has, out, float(out[has].mean()) if has.any() else 0.5).astype(np.float32)
INPUTS = [("normal", 0), ("native cleaned", 2)] if os.environ.get("EF_INPUTS", "both") == "both" else [("native cleaned", 2)]
def inliers_of(pairs):
    pa, pb = pairs
    if len(pa) < 4: return 0
    matrix, inl = fine_matching.fit_similarity(pa.astype(np.float32), pb.astype(np.float32)); return int(inl.sum()) if matrix is not None else 0

In [4]:
import time, matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
g = settings.GEOMETRY
MIN_HEIGHT = float(os.environ.get("EF_MIN_HEIGHT", 1000)); EVERY = int(os.environ.get("EF_EVERY", 1)); LIMIT = int(os.environ.get("EF_LIMIT", 0))
def level_like_photo(height, camera):
    gsd = height * g["scale"] / camera["fx"]
    return min((abs(math.log(level_gsd(L) / gsd)), L) for L in store.levels)[1]
def ground_grid(pose, camera, heading):
    pitch = roll = 0.0
    if g["attitude"]: pitch = g["pitch_sign"] * pose["pitch"]; roll = g["roll_sign"] * pose["roll"]
    ax, ay, az = photo.camera_axes(pose["yaw"] + g["yaw_offset"] + heading, pitch, roll)
    height = (pose["height"] + g["height_offset"]) * g["scale"]
    intrinsic = np.array([[camera["fx"], 0, camera["cx"]], [0, camera["fy"], camera["cy"]], [0, 0, 1]], float)
    dist = np.array([camera["k1"], camera["k2"], camera["p1"], camera["p2"], camera["k3"]])
    cols, rows = np.meshgrid(np.arange(320), np.arange(256)); pts = np.stack([cols, rows], -1).reshape(-1, 1, 2).astype(np.float64)
    und = cv2.undistortPointsIter(pts, intrinsic, dist, None, None, (cv2.TERM_CRITERIA_COUNT + cv2.TERM_CRITERIA_EPS, 50, 1e-10)).reshape(-1, 2)
    ray = und[:, :1] * ax + und[:, 1:] * ay + az; distance = -height / ray[:, 2]
    return (ray[:, 0] * distance).reshape(256, 320), (ray[:, 1] * distance).reshape(256, 320)
def draw_map_in_view(level, grid, x_m, y_m):
    """The map level drawn into the photo's 320 x 256 view for a camera whose nadir point is (x_m, y_m). Returns grey 0..1 and the share of pixels that have map."""
    east, north = grid; gsd = level_gsd(level); image = store.levels[level]
    map_x = ((x_m + east) / gsd - 0.5).astype(np.float32); map_y = ((y_m - north) / gsd - 0.5).astype(np.float32)
    x0 = int(max(map_x.min() - 4, 0)); x1 = int(min(map_x.max() + 5, image.shape[1])); y0 = int(max(map_y.min() - 4, 0)); y1 = int(min(map_y.max() + 5, image.shape[0]))
    if x1 <= x0 + 8 or y1 <= y0 + 8: return None, 0.0
    window = np.asarray(image[y0:y1, x0:x1]).astype(np.float32) / 255
    out = cv2.remap(window, map_x - x0, map_y - y0, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    has = cv2.remap((window > 0).astype(np.float32), map_x - x0, map_y - y0, cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    return out * (has > 0.99), float((has > 0.99).mean())
def photo_in_own_view(path, camera):
    frame = photo.load_photo(path)
    frame = flat_field.remove_vignette(frame, camera["cx"], camera["cy"], 3); frame = flat_field.local_normalise(frame, 15.0)
    return photo.normalise_frame(frame)
def wrong_places(truth):
    return [(truth[0] + dx, truth[1] + dy) for dx, dy in ((3000.0, 0.0), (-3000.0, 0.0), (0.0, 3000.0), (0.0, -3000.0))]
def up2(a): return cv2.resize(a, (a.shape[1] * 2, a.shape[0] * 2), interpolation=cv2.INTER_CUBIC)
query_list = [q for q in queries.query_list() if abs(q["scale_factor"] - 1.0) < 1e-6]
records = []; examples = []
todo = [i for i, r in enumerate(rows) if float(r["alt"]) >= MIN_HEIGHT][::EVERY]
if LIMIT: todo = todo[:LIMIT]
print(len(todo), "photos from", MIN_HEIGHT, "m")

130 photos from 1000.0 m


In [5]:
for index in todo:
    meta = rows[index]; camera = photo.read_camera(meta); pose = photo.read_pose(meta); path = os.path.join(DATASET, "images", meta["filename"])
    truth = store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"])); places = [truth] + wrong_places(truth)
    # --- arm 1: canvas (today)
    level = level_for_height(pose["height"], camera); size = settings.TILE
    with_settings(None); picture = photo.prepare_photo(path, pose, camera, level)
    canvases = [fine_matching.grey_canvas(*photo.draw_canvas(picture, pose, camera, level, q["heading"], q["scale_factor"])) for q in query_list]
    tiles = [map_piece(store.levels[level], level, x, y, size) for x, y in places]
    ok = [(t > 0).mean() > 0.9 for t in tiles]
    # --- arms 2 and 3: the photo as it is, the map drawn into its view
    lv = level_like_photo(pose["height"], camera); own = photo_in_own_view(path, camera)
    views = {}
    for q in query_list:
        grid = ground_grid(pose, camera, q["heading"])
        for k, (x, y) in enumerate(places):
            drawn, share = draw_map_in_view(lv, grid, x, y)
            if drawn is None or share < 0.95: ok[k] = False; continue
            views[(q["heading"], k)] = clean_tile(drawn, 15.0)
    if not ok[0] or sum(ok[1:]) == 0: continue
    for name, engine in engines.items():
        for inp in ("canvas", "photo view", "photo view x2"):
            true_best = 0; null_best = [0] * 4
            for q_i, q in enumerate(query_list):
                for k in range(5):
                    if not ok[k]: continue
                    if inp == "canvas": a, b = canvases[q_i], tiles[k]
                    else:
                        if (q["heading"], k) not in views: continue
                        a, b = own, views[(q["heading"], k)]
                        if inp == "photo view x2": a, b = up2(a), up2(b)
                    n = inliers_of(engine(a, b))
                    if k == 0: true_best = max(true_best, n)
                    else: null_best[k - 1] = max(null_best[k - 1], n)
            records.append(dict(photo=index, height=pose["height"], input=inp, matcher=name, true_inl=true_best, null_inl=[n for n, o in zip(null_best, ok[1:]) if o]))
    if len(examples) < 4 and pose["height"] >= 1300: examples.append((own, views.get((0.0, 0)), views.get((0.0, 1)), canvases[1], tiles[0], pose["height"]))
    print(index, flush=True)
tab = pd.DataFrame(records); tab.to_pickle(os.path.join(OUT, "photo_view%s.pkl" % os.environ.get("EF_TAG", ""))); tab.assign(null_inl=tab.null_inl.astype(str)).to_csv(os.path.join(OUT, "photo_view%s.csv" % os.environ.get("EF_TAG", "")), index=False)
if examples:
    fig, ax = plt.subplots(len(examples), 5, figsize=(15, 3 * len(examples)))
    for r, (own, tv, wv, cv_, tt, h) in enumerate(examples):
        for c, (im, t) in enumerate(((own, "photo, cleaned, as it is"), (tv, "map drawn in its view (true place)"), (wv, "map in its view (wrong place, 3 km)"), (cv_, "today: photo on 224 px canvas"), (tt, "today: true map tile"))):
            ax[r, c].imshow(im if im is not None else np.zeros((8, 8)), cmap="gray"); ax[r, c].axis("off"); ax[r, c].set_title("%s\n%.0f m" % (t, h) if c == 0 else t, fontsize=8)
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "photo_view_examples%s.png" % os.environ.get("EF_TAG", "")), dpi=110)
print(len(tab), "records")

/home/workstation/training/ideaforge/venv-geoloc/lib/python3.10/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


2


3


4


10


14


... (122 outputs left out)

325


329


331


780 records


In [6]:
from sklearn.metrics import roc_auc_score
tab["band"] = pd.cut(tab.height, [MIN_HEIGHT - 1, 1300, 1600, 2100]); tab["fold"] = tab.photo % 2
def nulls_of(g): return np.concatenate([np.array(v, float) for v in g.null_inl if len(v)])
def cv_recall(all_g, sel_g):
    hits = []
    for fold in (0, 1):
        bar = nulls_of(all_g[all_g.fold != fold]).max(); s = sel_g[sel_g.fold == fold]; hits += list((s.true_inl > bar).astype(float))
    return float(np.mean(hits)) if hits else np.nan
out = []
for (m, i), g_all in tab.groupby(["matcher", "input"]):
    t = g_all.true_inl.values.astype(float); n = nulls_of(g_all)
    out.append(dict(matcher=m, input=i, photos=len(g_all), true_median=np.median(t), null_median=np.median(n), null_max=n.max(), auc=round(roc_auc_score(np.r_[np.ones(len(t)), np.zeros(len(n))], np.r_[t, n]), 3), recall_cv=round(cv_recall(g_all, g_all), 3)))
print("all photos from %d m: bars from each input's own wrong-place counts" % MIN_HEIGHT); print(pd.DataFrame(out).to_string(index=False))
band_rows = []
for (b, m, i), g in tab.groupby(["band", "matcher", "input"], observed=True):
    g_all = tab[(tab.matcher == m) & (tab.input == i)]; n = nulls_of(g); t = g.true_inl.values.astype(float)
    band_rows.append(dict(band=str(b), matcher=m, input=i, n=len(g), true_median=np.median(t), recall_cv=round(cv_recall(g_all, g), 2), auc=round(roc_auc_score(np.r_[np.ones(len(t)), np.zeros(len(n))], np.r_[t, n]), 2) if len(g) > 2 and len(n) else np.nan))
bt = pd.DataFrame(band_rows); print("\nby height band"); print(bt.pivot_table(index="band", columns=["matcher", "input"], values=["recall_cv", "auc", "true_median"]).round(2).to_string())
bt.to_csv(os.path.join(OUT, "photo_view_bands%s.csv" % os.environ.get("EF_TAG", "")), index=False)
print("\nkill rule: photo-view input beats canvas by >= 0.10 recall (same matcher) in how many of the 3 bands")
for m in tab.matcher.unique():
    base = bt[(bt.matcher == m) & (bt.input == "canvas")].set_index("band").recall_cv
    for inp in ("photo view", "photo view x2"):
        cur = bt[(bt.matcher == m) & (bt.input == inp)].set_index("band").recall_cv
        wins = int(((cur - base) >= 0.10).sum()); print("  %s %-14s bands won: %d of %d  -> %s" % (m, inp, wins, len(base), "USEFUL" if wins >= 2 else "killed"))

all photos from 1000 m: bars from each input's own wrong-place counts
matcher         input  photos  true_median  null_median  null_max   auc  recall_cv
  loftr        canvas     130          0.0          0.0       5.0 0.733      0.431
  loftr    photo view     130         15.0          0.0       6.0 0.957      0.700
  loftr photo view x2     130          0.0          0.0       4.0 0.661      0.315
  xoftr        canvas     130         14.0          0.0       9.0 0.899      0.600
  xoftr    photo view     130         77.0          4.0      20.0 0.999      0.954
  xoftr photo view x2     130          8.5          0.0       0.0 0.846      0.692

by height band
                    auc                                                          recall_cv                                                          true_median                                                         
matcher           loftr                           xoftr                              loftr                          